# ZTLF 04 — Downstream model impact and subgroup fairness

The original write-up called governed data "AI-ready" and never actually trained a model on it. Time to test that.

Protocol: corruption only touches the training set. Every condition gets scored on the same clean, held-out test set.

This matters more than it sounds like it should. If you corrupt the whole dataset and split afterward, the test set is dirty too, and quarantine looks better than it is by deleting the hard test rows. That's leakage, not a real data-quality effect.

| condition | meaning |
|---|---|
| clean | train on uncorrupted data (upper bound) |
| corrupted | train on corrupted data, no gate (lower bound) |
| quarantine | drop every row with a flagged cell (my original default) |
| repair | normalise representation, null the rest, impute downstream |

Metrics: ROC-AUC and PR-AUC for discrimination, Brier score for calibration (which can quietly get worse even when AUC looks fine), and `n_train`, since that's the real cost of a policy.

Early result worth flagging: on bank marketing at 10% contamination, quarantine threw away half the training rows and left gradient boosting worse off than doing nothing. Logistic regression benefited, though. So this is model-dependent, and I should report it that way — "quarantine improves the quality metric while hurting the model" is a more useful and more honest finding than a blanket claim that governance helps.


In [ ]:
#@title Mount and load
import pathlib, sys, importlib, warnings, dataclasses, time
warnings.filterwarnings("ignore")
from google.colab import drive
drive.mount('/content/drive')
PROJECT_ROOT = pathlib.Path("/content/drive/MyDrive/Paper1")
sys.path.insert(0, str(PROJECT_ROOT/"src"))
TABLES=PROJECT_ROOT/"outputs/tables"; FIGURES=PROJECT_ROOT/"outputs/figures"
METRICS=PROJECT_ROOT/"outputs/metrics"
for d in (TABLES,FIGURES,METRICS): d.mkdir(parents=True, exist_ok=True)

required=["ztlf_profiling.py","ztlf_specs.py","ztlf_corruption.py",
          "ztlf_plans.py","ztlf_baselines.py","ztlf_downstream.py"]
missing=[f for f in required if not (PROJECT_ROOT/"src"/f).exists()]
if missing:
    raise SystemExit(f"Missing modules: {missing}. Copy them into src/. "
                     "Never generate replacements.")

import pandas as pd, numpy as np
import ztlf_profiling, ztlf_specs, ztlf_corruption, ztlf_plans, ztlf_baselines, ztlf_downstream
for m in (ztlf_profiling,ztlf_specs,ztlf_corruption,ztlf_plans,ztlf_baselines,ztlf_downstream):
    importlib.reload(m)
from ztlf_profiling import load_dataset
from ztlf_specs import *
from ztlf_corruption import assign_row_ids, ROW_ID
from ztlf_plans import PLANS, seed_list
from ztlf_baselines import RuleSet, gate_ztlf
from ztlf_downstream import *
print("modules OK")

In [ ]:
#@title Corpora, tasks and TWO rule sets (naive vs census-informed)
RAW = PROJECT_ROOT/"data/raw"
SPECS = {"bank_marketing_full": bank_spec("bank_marketing_full", str(RAW/"bank-full.csv")),
         "diabetes_130us": dataclasses.replace(DIABETES_SPEC, path=str(RAW/"diabetic_data.csv"))}

CLEAN = {}
for n, s in SPECS.items():
    if pathlib.Path(s.path).exists():
        CLEAN[n] = add_derived_subgroups(assign_row_ids(load_dataset(s), n[:4]))
        print(n, len(CLEAN[n]))

DIAB_DRUGS = {c: DIAB_DRUG_LEVELS for c in ("metformin","insulin","glipizide","glyburide")}

_bank_common = dict(
    domains={"job":BANK_JOBS,"marital":BANK_MARITAL,"education":BANK_EDUCATION,
             "month":BANK_MONTH,"contact":BANK_CONTACT,"poutcome":BANK_POUTCOME,
             "default":BANK_BINARY,"housing":BANK_BINARY,"loan":BANK_BINARY,"y":BANK_BINARY},
    ranges={"age":(18,100),"duration":(0,5000),"balance":(-10000,200000),
            "day":(1,31),"campaign":(1,100)},
    sentinels=("unknown",))

_diab_common = dict(
    domains={"gender":DIAB_GENDER,"readmitted":DIAB_READMIT,
             "change":{"Ch","No"},"diabetesMed":{"Yes","No"}, **DIAB_DRUGS},
    ranges={"time_in_hospital":(1,14),"num_medications":(1,100),
            "num_lab_procedures":(0,200),"number_diagnoses":(1,20)},
    sentinels=("?",))

# Candidate completeness rules a practitioner would plausibly write
BANK_NN_CANDIDATES = ["job","education","contact","poutcome"]
DIAB_NN_CANDIDATES = ["race","weight","payer_code","medical_specialty"]

# --- census-informed authoring: drop not-null rules on columns that are
# --- missing by design. This is the framework consulting its own Phase 1 output.
REJECTED = {}
NN_KEPT = {}
for name, cands, sent in (("bank_marketing_full", BANK_NN_CANDIDATES, ("unknown",)),
                          ("diabetes_130us", DIAB_NN_CANDIDATES, ("?",))):
    if name not in CLEAN:
        continue
    kept, rej = census_informed_not_null(CLEAN[name], cands, sentinels=sent,
                                         max_natural_missing_pct=5.0)
    NN_KEPT[name], REJECTED[name] = kept, rej
    print(f"\n{name}\n  kept not-null rules : {kept}"
          f"\n  rejected by census  : {rej}")

RULE_SETS = {
  "naive": {
    "bank_marketing_full": RuleSet("bank_marketing_full",
        not_null=BANK_NN_CANDIDATES, **_bank_common),
    "diabetes_130us": RuleSet("diabetes_130us",
        not_null=DIAB_NN_CANDIDATES, **_diab_common)},
  "census_informed": {
    "bank_marketing_full": RuleSet("bank_marketing_full",
        not_null=NN_KEPT.get("bank_marketing_full", []), **_bank_common),
    "diabetes_130us": RuleSet("diabetes_130us",
        not_null=NN_KEPT.get("diabetes_130us", []), **_diab_common)},
}

# Table 8b -- which rules the census removed and why (a manuscript table)
T8b = pd.DataFrame([
    {"dataset": d, "rejected_rule": f"not_null({c})", "natural_missing_pct": p}
    for d, rej in REJECTED.items() for c, p in rej.items()])
if len(T8b):
    T8b.to_csv(TABLES/"T8b_census_rejected_rules.csv", index=False)
T8b


# -----------------------------------------------------------------------------

In [ ]:
#@title Fixed clean train/test split  (test set NEVER corrupted)
from sklearn.model_selection import train_test_split
SPLITS={}
for name, df in CLEAN.items():
    task=TASKS[name]()
    y=task.positive_label(df[task.target])
    tr,te=train_test_split(df, test_size=0.25, random_state=42, stratify=y)
    SPLITS[name]=(tr.reset_index(drop=True), te.reset_index(drop=True))
    print(f"{name}: train={len(tr):,} test={len(te):,} "
          f"test positive rate={task.positive_label(te[task.target]).mean():.3f}")

In [ ]:
#@title Run the downstream experiment (both rule sets)
RATES = [0.05, 0.10, 0.20]   #@param
N_SEEDS = 5                  #@param {type:"integer"}
SEEDS = seed_list(20260803, N_SEEDS)
MODELS = ["logreg", "hgb"]

rows, sub_rows = [], []
t0 = time.time()
for rs_name, rs_map in RULE_SETS.items():
    for name, (tr, te) in SPLITS.items():
        task = TASKS[name]()
        rules = rs_map[name]
        for rate in RATES:
            for seed in SEEDS:
                cor, _ = PLANS[name]().run(tr, rate, seed)
                cor = add_derived_subgroups(cor)
                det = gate_ztlf(cor, rules)
                conds = {"clean": tr, "corrupted": cor,
                         "quarantine": policy_quarantine(cor, det),
                         "repair": policy_repair(cor, det, task)}
                for cname, cdf in conds.items():
                    for mdl in MODELS:
                        r = evaluate_condition(cdf, te, task, cname, mdl, seed)
                        p = r.pop("_pred", None); yt = r.pop("_ytrue", None)
                        r.update(dataset=name, rate=rate, rule_set=rs_name)
                        rows.append(r)
                        if p is not None and mdl == "hgb":
                            # prevalence-matched threshold (see module docstring)
                            sg = subgroup_metrics(te, yt, p, task.subgroup_columns)
                            sg["dataset"] = name; sg["condition"] = cname
                            sg["rate"] = rate; sg["seed"] = seed
                            sg["rule_set"] = rs_name
                            sub_rows.append(sg)
            print(f"  [{rs_name}] {name} rate {rate} done ({time.time()-t0:.0f}s)")

down = pd.DataFrame(rows)
subs = pd.concat(sub_rows, ignore_index=True)
down.to_csv(METRICS/"downstream_raw.csv", index=False)
subs.to_csv(METRICS/"downstream_subgroups_raw.csv", index=False)
print(f"\n{len(down)} model runs in {time.time()-t0:.0f}s")


# -----------------------------------------------------------------------------

In [ ]:
#@title Table 9 — downstream impact, by rule set
T9 = (down[down.status=="ok"]
      .groupby(["dataset","rule_set","rate","model","condition"])
      .agg(roc_auc=("roc_auc","mean"), roc_sd=("roc_auc","std"),
           pr_auc=("pr_auc","mean"), brier=("brier","mean"),
           n_train=("n_train","mean")).round(4).reset_index())
T9["pct_train_retained"] = (100*T9.n_train /
    T9.groupby(["dataset","rule_set","rate","model"]).n_train.transform("max")).round(1)
T9.to_csv(TABLES/"T9_downstream_impact.csv", index=False)
T9


# -----------------------------------------------------------------------------
# NEW CELL: insert after Table 9
# -----------------------------------------------------------------------------
#@title Table 9b — RULE AUTHORSHIP dominates rule enforcement (headline)
q = T9[(T9.condition=="quarantine")]
T9b = (q.pivot_table(index=["dataset","rate","model"], columns="rule_set",
                     values=["roc_auc","pct_train_retained"])
       .round(4).reset_index())
T9b.to_csv(TABLES/"T9b_rule_authorship_effect.csv", index=False)
print("Same framework, same enforcement engine, same data.\n"
      "Only the ORIGIN of the completeness rules differs:\n"
      "  naive           = rules written without profiling the data\n"
      "  census_informed = not-null rules dropped where the column is\n"
      "                    missing by documented design (>5% natural)\n")
T9b


# -----------------------------------------------------------------------------

In [ ]:
#@title Table 9b — RULE AUTHORSHIP dominates rule enforcement (headline)
q = T9[(T9.condition=="quarantine")]
T9b = (q.pivot_table(index=["dataset","rate","model"], columns="rule_set",
                     values=["roc_auc","pct_train_retained"])
       .round(4).reset_index())
T9b.to_csv(TABLES/"T9b_rule_authorship_effect.csv", index=False)
print("Same framework, same enforcement engine, same data.\n"
      "Only the ORIGIN of the completeness rules differs:\n"
      "  naive           = rules written without profiling the data\n"
      "  census_informed = not-null rules dropped where the column is\n"
      "                    missing by documented design (>5% natural)\n")
T9b


# -----------------------------------------------------------------------------
# REPLACE CELL: "Table 10"
# -----------------------------------------------------------------------------
#@title Table 10 — does the gate beat doing nothing?
base = (down[(down.status=="ok")&(down.condition=="corrupted")]
        .groupby(["dataset","rule_set","rate","model"]).roc_auc.mean()
        .rename("corrupted_auc"))
comp = (down[(down.status=="ok")&(down.condition.isin(["quarantine","repair","clean"]))]
        .groupby(["dataset","rule_set","rate","model","condition"]).roc_auc.mean()
        .rename("auc"))
T10 = comp.reset_index().merge(base.reset_index(),
                               on=["dataset","rule_set","rate","model"])
T10["delta_auc"] = (T10.auc - T10.corrupted_auc).round(4)
T10["helps"] = np.where(T10.delta_auc > 0, "yes", "NO")
T10.to_csv(TABLES/"T10_gate_benefit.csv", index=False)
T10


# -----------------------------------------------------------------------------
# REPLACE CELL: "Table 11"
# -----------------------------------------------------------------------------
#@title Table 11 — subgroup recall (prevalence-matched threshold)
T11 = (subs.groupby(["dataset","rule_set","condition","subgroup_column","subgroup"])
       .agg(recall=("recall","mean"), recall_sd=("recall","std"),
            n_test=("n_test","mean"), base_rate=("base_rate","mean"))
       .round(4).reset_index())
T11.to_csv(TABLES/"T11_subgroup_recall.csv", index=False)

gap = (T11.groupby(["dataset","rule_set","condition","subgroup_column"])
       .agg(max_recall=("recall","max"), min_recall=("recall","min")).reset_index())
gap["recall_gap"] = (gap.max_recall - gap.min_recall).round(4)
gap.to_csv(TABLES/"T11b_subgroup_recall_gap.csv", index=False)
print("Recall is now measured at a prevalence-matched operating point, not a\n"
      "fixed 0.5 cut-off. With an ~11% base rate a 0.5 threshold predicted\n"
      "almost no positives, collapsing every subgroup recall toward zero and\n"
      "making the fairness comparison uninformative.\n")
gap


# -----------------------------------------------------------------------------
# REPLACE CELL: "Figure 5"
# -----------------------------------------------------------------------------
#@title Figure 5 — accuracy vs data retained, by rule set
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

ok = T9[T9.model=="hgb"]
ds = sorted(ok.dataset.unique())
fig, axes = plt.subplots(1, len(ds), figsize=(6.4*len(ds), 4.8), squeeze=False)
markers = {"clean":"*", "corrupted":"o", "quarantine":"s", "repair":"^"}
colors  = {"naive":"tab:red", "census_informed":"tab:blue"}
for ax, name in zip(axes[0], ds):
    d = ok[ok.dataset==name]
    for rs in d.rule_set.unique():
        for cond, mk in markers.items():
            s = d[(d.condition==cond)&(d.rule_set==rs)]
            ax.scatter(s.pct_train_retained, s.roc_auc, s=95, marker=mk,
                       color=colors.get(rs,"gray"), alpha=.8,
                       label=f"{rs}/{cond}")
    ax.set_xlabel("% of training rows retained"); ax.set_ylabel("ROC-AUC")
    ax.set_title(name); ax.grid(alpha=.3)
axes[0][0].legend(fontsize=6, ncol=2, loc="lower right")
fig.suptitle("Rule authorship determines whether quality gating helps or destroys the corpus", y=1.02)
fig.tight_layout(); fig.savefig(FIGURES/"F5_downstream_tradeoff.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
#@title Table 10 — does the gate beat doing nothing?
base = (down[(down.status=="ok")&(down.condition=="corrupted")]
        .groupby(["dataset","rule_set","rate","model"]).roc_auc.mean()
        .rename("corrupted_auc"))
comp = (down[(down.status=="ok")&(down.condition.isin(["quarantine","repair","clean"]))]
        .groupby(["dataset","rule_set","rate","model","condition"]).roc_auc.mean()
        .rename("auc"))
T10 = comp.reset_index().merge(base.reset_index(),
                               on=["dataset","rule_set","rate","model"])
T10["delta_auc"] = (T10.auc - T10.corrupted_auc).round(4)
T10["helps"] = np.where(T10.delta_auc > 0, "yes", "NO")
T10.to_csv(TABLES/"T10_gate_benefit.csv", index=False)
T10


# -----------------------------------------------------------------------------

In [ ]:
#@title Table 11 — subgroup recall (prevalence-matched threshold)
T11 = (subs.groupby(["dataset","rule_set","condition","subgroup_column","subgroup"])
       .agg(recall=("recall","mean"), recall_sd=("recall","std"),
            n_test=("n_test","mean"), base_rate=("base_rate","mean"))
       .round(4).reset_index())
T11.to_csv(TABLES/"T11_subgroup_recall.csv", index=False)

gap = (T11.groupby(["dataset","rule_set","condition","subgroup_column"])
       .agg(max_recall=("recall","max"), min_recall=("recall","min")).reset_index())
gap["recall_gap"] = (gap.max_recall - gap.min_recall).round(4)
gap.to_csv(TABLES/"T11b_subgroup_recall_gap.csv", index=False)
print("Recall is now measured at a prevalence-matched operating point, not a\n"
      "fixed 0.5 cut-off. With an ~11% base rate a 0.5 threshold predicted\n"
      "almost no positives, collapsing every subgroup recall toward zero and\n"
      "making the fairness comparison uninformative.\n")
gap


# -----------------------------------------------------------------------------

In [ ]:
#@title Figure 5 — accuracy vs data retained, by rule set
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

ok = T9[T9.model=="hgb"]
ds = sorted(ok.dataset.unique())
fig, axes = plt.subplots(1, len(ds), figsize=(6.4*len(ds), 4.8), squeeze=False)
markers = {"clean":"*", "corrupted":"o", "quarantine":"s", "repair":"^"}
colors  = {"naive":"tab:red", "census_informed":"tab:blue"}
for ax, name in zip(axes[0], ds):
    d = ok[ok.dataset==name]
    for rs in d.rule_set.unique():
        for cond, mk in markers.items():
            s = d[(d.condition==cond)&(d.rule_set==rs)]
            ax.scatter(s.pct_train_retained, s.roc_auc, s=95, marker=mk,
                       color=colors.get(rs,"gray"), alpha=.8,
                       label=f"{rs}/{cond}")
    ax.set_xlabel("% of training rows retained"); ax.set_ylabel("ROC-AUC")
    ax.set_title(name); ax.grid(alpha=.3)
axes[0][0].legend(fontsize=6, ncol=2, loc="lower right")
fig.suptitle("Rule authorship determines whether quality gating helps or destroys the corpus", y=1.02)
fig.tight_layout(); fig.savefig(FIGURES/"F5_downstream_tradeoff.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
#@title Figure 6 — subgroup recall gap by policy
fig,ax=plt.subplots(figsize=(9,4.5))
g=gap[gap.dataset==gap.dataset.iloc[0]] if len(gap) else gap
piv=gap.pivot_table(index="condition", columns="subgroup_column",
                    values="recall_gap", aggfunc="mean")
piv.plot(kind="bar", ax=ax)
ax.set_ylabel("recall gap (max - min across subgroups)")
ax.set_title("Disparate impact of quality policies\n(lower is fairer)")
ax.grid(axis="y", alpha=.3); ax.tick_params(axis="x", rotation=0)
fig.tight_layout(); fig.savefig(FIGURES/"F6_subgroup_gap.png",dpi=300)
plt.show()

In [ ]:
#@title Theta sensitivity — add as a new cell at the end of notebook 04
# Sweeps the admission threshold across its full range, including theta = 100
# (which admits every candidate rule and is therefore identical to the naive
# rule set). Establishes whether the Section 5.5 result depends on the
# specific value of theta or holds across a plateau.

import time
import pandas as pd, numpy as np

THETAS = [0, 1, 5, 10, 20, 30, 50, 100]   #@param
THETA_RATE = 0.10                          #@param {type:"number"}
THETA_SEEDS = SEEDS[:5]

# candidate completeness rules per corpus, and the sentinel that marks absence
THETA_CFG = {
    "bank_marketing_full": dict(cands=BANK_NN_CANDIDATES, sent=("unknown",),
                                common=_bank_common),
    "diabetes_130us":      dict(cands=DIAB_NN_CANDIDATES, sent=("?",),
                                common=_diab_common),
}

rows = []
t0 = time.time()
for name, (tr, te) in SPLITS.items():
    cfg = THETA_CFG[name]
    task = TASKS[name]()
    for th in THETAS:
        kept, rejected = census_informed_not_null(
            tr, cfg["cands"], sentinels=cfg["sent"], max_natural_missing_pct=th)
        rules = RuleSet(name, not_null=kept, **cfg["common"])
        for seed in THETA_SEEDS:
            cor, _ = PLANS[name]().run(tr, THETA_RATE, seed)
            cor = add_derived_subgroups(cor)
            q = policy_quarantine(cor, gate_ztlf(cor, rules))
            r = evaluate_condition(q, te, task, "quarantine", "hgb", seed)
            rows.append(dict(dataset=name, theta=th, n_rules=len(kept),
                             admitted=";".join(kept) or "(none)",
                             seed=seed,
                             retained_pct=round(100*len(q)/len(cor), 2),
                             auc=r["roc_auc"]))
    print(f"  {name} done ({time.time()-t0:.0f}s)")

theta_raw = pd.DataFrame(rows)
theta_raw.to_csv(METRICS/"theta_sensitivity_raw.csv", index=False)

T12 = (theta_raw.groupby(["dataset", "theta"])
       .agg(n_rules=("n_rules", "first"), admitted=("admitted", "first"),
            retained_pct=("retained_pct", "mean"),
            auc=("auc", "mean"), auc_sd=("auc", "std"))
       .round(4).reset_index())
T12.to_csv(TABLES/"T12_theta_sensitivity.csv", index=False)
print("\nRetention and AUC should be flat across a plateau containing theta=5.")
T12

In [ ]:
#@title Figure 7 — theta sensitivity
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

ds = sorted(T12.dataset.unique())
fig, axes = plt.subplots(1, len(ds), figsize=(6.2*len(ds), 4.2), squeeze=False)
for ax, name in zip(axes[0], ds):
    d = T12[T12.dataset == name].sort_values("theta")
    ax2 = ax.twinx()
    ax.plot(d.theta, d.retained_pct, marker="o", color="tab:blue",
            label="training rows retained (%)")
    ax2.plot(d.theta, d.auc, marker="s", color="tab:orange", label="ROC-AUC")
    ax.axvline(5, ls="--", color="gray", lw=1)
    ax.text(5.6, ax.get_ylim()[1]*0.95, "θ = 5% (adopted)", fontsize=8,
            color="gray", va="top")
    ax.set_xlabel("admission threshold θ (% natural missingness)")
    ax.set_ylabel("training rows retained (%)", color="tab:blue")
    ax2.set_ylabel("ROC-AUC", color="tab:orange")
    ax.set_title(name); ax.grid(alpha=.3)
fig.suptitle("Admission threshold sensitivity: the result holds across a plateau", y=1.02)
fig.tight_layout()
fig.savefig(FIGURES/"F7_theta_sensitivity.png", dpi=300, bbox_inches="tight")
plt.show()

---
### Notes for the writeup

Don't claim governance unconditionally improves AI outcomes — the numbers don't back that up. What I can actually say:

- Quarantine is expensive. At 10% contamination it dropped roughly half the training rows. Always report retention next to accuracy — a policy that throws away half the data for a 0.006 AUC gain is a bad trade, and a referee will catch it if I don't mention it myself.
- The benefit is model-dependent. Cleaning helped logistic regression, hurt gradient boosting (which shrugs off the injected noise anyway). Lines up with CleanML (Li et al., ICDE 2021) and Mohammed et al. (Information Systems, 2025) — cite both.
- Repair usually beats quarantine, since it keeps the row and just blanks the bad value. If that holds up, the framework's default policy should change — and saying so directly is a strength, not an admission. A framework that measures its own choices and revises them is doing exactly what it's supposed to.
- Report the subgroup gap. If quarantine widens the recall gap between groups, that's a real governance cost that never shows up in an aggregate metric.

Net effect: the contribution isn't "this framework makes AI-ready data." It's "governance decisions have measurable costs, sometimes negative ones, and this framework is what makes them visible instead of hidden." Smaller claim, much more defensible.

### Still on the list before submission
- Redraw Figures 1–3 without generative AI (Elsevier's policy on this is explicit)
- Fix the affiliation mismatch across the cover letter, title page, and bio
- Archive the repo to Zenodo for a DOI
- Consider SOTA cleaning baselines (Raha, Baran, HoloClean) if targeting DKE instead
